# SoftVertexEdgeStitch – StVK membrane symbol derivation

Each (vertex, edge) pair forms a triangle: 1 vertex from mesh A and 2 vertices of an edge from mesh B.
We use **StVK (Saint-Venant–Kirchhoff) membrane** energy, which is polynomial (no log singularity).

- Current metric tensor: A = [[e01·e01, e01·e02], [e02·e01, e02·e02]]
- Right Cauchy-Green deformation: C = IB * A
- Green-Lagrange strain: E_G = (C - I) / 2
- Energy: Ψ = μ * tr(E_G²) + λ/2 * tr(E_G)²
- Output: A, E, dEdX, ddEddX for use in inter_primitive backend.

In [ ]:
import sys
import os
sys.path.append('../')
import pathlib as pl
from SymEigen import *

from sympy import symbols, Matrix, eye, Rational
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()

lmbd, mu = symbols("lambda mu")

X = Eigen.Vector("X", 9)
X0 = X[0:3,0]
X1 = X[3:6,0]
X2 = X[6:9,0]

E01 = X1 - X0
E02 = X2 - X0

A = Matrix(
[
    [E01.dot(E01), E01.dot(E02)],
    [E02.dot(E01), E02.dot(E02)]
])

A = Eigen.FromSympy("A", A)

In [ ]:
IB = Eigen.Matrix("IB", 2, 2)

In [ ]:
C = IB * A
I2 = eye(2)
E_G = (C - I2) / 2
E_G

In [ ]:
E = mu * (E_G * E_G).trace() + Rational(1, 2) * lmbd * (E_G.trace())**2
E

In [ ]:
dEdX = VecDiff(E, X)
dEdX

In [ ]:
ddEddX = VecDiff(dEdX, X)
ddEddX

In [ ]:
ClA = Gen.Closure(X)
ClE = Gen.Closure(lmbd, mu, X, IB)

In [ ]:
s = f'''
{ClA("A",A)}
{ClE("E",E)}
{ClE("dEdX",dEdX)}
{ClE("ddEddX",ddEddX)}
'''
print(s)

f = open( backend_source_dir('cuda') / 'inter_primitive_effect_system/constitutions/details/soft_vertex_edge_stitch.inl', 'w')
f.write(s)
f.close()